### theprotocol.it - scrape all records
### WORKING

In [1]:
import httpx

def get_protocol_csrf_token():
    """Get XSRF-TOKEN cookie from /csrf-token endpoint."""
    url = "https://apus-api.theprotocol.it/csrf-token"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    with httpx.Client() as client:
        client.get(url, headers=headers)
        # print(client.cookies)
        return client.cookies
    
def fetch_protocol_offers(page_number=80, page_size=50):
    """Fetch job offers from theprotocol.it API for a given page."""
    url = f"https://apus-api.theprotocol.it/offers/_search?pageNumber={page_number}&orderby.field=Relevance&pageSize={page_size}"
    cookies = get_protocol_csrf_token()
    xsrf_token = cookies.get("XSRF-TOKEN")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Content-Type": "application/json",
        "x-xsrf-token": xsrf_token
    }
    with httpx.Client(cookies=cookies) as client:
        response = client.post(url, headers=headers)

        if not response.text:
            return None
        return response.json()['offers']

offers = fetch_protocol_offers()
print(len(offers))
# import json
# offers_json = json.dumps(offers, indent=4, ensure_ascii=False)
# print(offers_json)

50


### rocketjobs.pl - scrape all records

In [2]:
import httpx
def fetch_justjoin_offers():
    """Fetch job offers from justjoin.it API."""
    url = "https://api.rocketjobs.pl/v2/user-panel/offers/by-cursor"
    params = {
        # "categories[]": 19,
        "currency": "pln",
        "from": 000,
        "itemsCount": 100,
        # "keywords[]": "python",
        "orderBy": "DESC",
        # "remoteWorkOptions[]": "remote",
        "sortBy": "published"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    response = httpx.get(url, params=params, headers=headers)
    offers = response.json()['data']
    return offers
offers = fetch_justjoin_offers()
print(f"Fetched {len(offers)} offers from rocketjobs.pl")
# print(offers)
# for offer in offers:
#     print(offer)

Fetched 100 offers from rocketjobs.pl


### justjoin.it - Scrape all records

In [3]:
import httpx
def fetch_justjoin_offers():
    """Fetch job offers from justjoin.it API."""
    url = "https://api.justjoin.it/v2/user-panel/offers/by-cursor"
    params = {
        # "categories[]": 19,
        "currency": "pln",
        "from": 3300,
        "itemsCount": 100,
        # "keywords[]": "python",
        "orderBy": "DESC",
        "remoteWorkOptions[]": "remote",
        "sortBy": "published"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7"
    }
    response = httpx.get(url, params=params, headers=headers)
    offers = response.json()['data']
    return offers
offers = fetch_justjoin_offers()
print(f"Fetched {len(offers)} offers from justjoin.it")
# print(offers)
# for offer in offers:
#     print(offer)

Fetched 19 offers from justjoin.it


#### Solidjobs - scrape all records - filter from url
Poprawnie filtruje wszystko

In [4]:
# test_link_all_params = "https://solid.jobs/offers/it;cities=Krak%C3%B3w,Praca%20zdalna,Warszawa;categories=Programista;experiences=Regular,Junior;minimumSalary=3000;subcategories=Python"
# test_link_all_params = "https://solid.jobs/offers/it;cities=Krak%C3%B3w,Praca%20zdalna,Warszawa;categories=Tester;experiences=Regular,Junior;minimumSalary=3000;subcategories=In%C5%BCynier%20test%C3%B3w%20automatycznych"
test_link_all_params = "https://solid.jobs/offers/it;experiences=Senior;cities=Tr%C3%B3jmiasto;categories=Analityk;minimumSalary=25500"
import urllib.parse
import httpx
def parse_url_params(url):
    """Parse filter parameters from URL."""
    params = {}
    parts = url.split(';')
    for part in parts:
        if '=' in part:
            key, value = part.split('=', 1)
            decoded_value = urllib.parse.unquote(value)
            if key == "cities" and "Trójmiasto" in decoded_value:
                decoded_value = decoded_value.replace("Trójmiasto", "Gdańsk,Gdynia,Sopot")
            params[key] = decoded_value
    print(params)
    return params

def filter_offers(offers, cities=None, categories=None, subcategories=None, experiences=None, minimumSalary=None):
    """Filter job offers by parameters."""
    filtered = []
    for offer in offers:
        if cities:
            city_list = cities.split(',')
            remote_values = ["W całości", "Możliwa w całości"]
            is_remote = offer.get("remotePossible") in remote_values
            city = offer.get("companyCity") in city_list
            
            is_fully_remote = "Praca zdalna" in city_list and is_remote
            invalid_location = not(is_fully_remote or city)
            if invalid_location:
                continue # skip record
        if categories and offer.get("mainCategory") != categories:
            continue # skip record
        if subcategories and offer.get("subCategory") != subcategories:
            continue # skip record
        if experiences:
            exp_list = experiences.split(',')
            if offer.get("experienceLevel") not in exp_list:
                continue # skip record
        salary = offer.get("salaryRange", {})

        is_salary_below_minimum = minimumSalary and (salary["upperBound"] < float(minimumSalary))
        if is_salary_below_minimum:
            continue # skip record
        filtered.append(offer)
    return filtered

def get_all_offers():
    solidjobs = 'https://solid.jobs/api/offers?division=it&sortOrder=default'
    headers = {
        "Accept": "application/vnd.solidjobs.jobofferlist+json, application/json, text/plain, */*",
    }
    solid = httpx.get(solidjobs, headers=headers)
    return solid.json()


params = parse_url_params(test_link_all_params)
offers = get_all_offers()
filtered_offers = filter_offers(offers, **params)
# import json
# print(json.dumps(filtered_offers, ensure_ascii=False, indent=4))
print(len(filtered_offers))
for offer in filtered_offers:
    print(offer['id'], offer['jobTitle'], offer['companyName'], offer['companyCity'], offer['experienceLevel'])

{'experiences': 'Senior', 'cities': 'Gdańsk,Gdynia,Sopot', 'categories': 'Analityk', 'minimumSalary': '25500'}
1
24017 Expert IT Analyst (Payments) emagine Gdańsk Senior


# NON-SELENIUM SITES - GET ALL RECORDS

In [ ]:
BULLDOGJOB = "https://bulldogjob.pl"
INHIRE = "https://inhire.io"

### NoFluffJobs.com - working

In [22]:
def fetch_nofluffjobs_offers(page=1, page_size=5000):
    """Fetch job offers from NoFluffJobs API."""
    url = "https://nofluffjobs.com/api/joboffers/main"
    params = {
        "pageTo": page,
        "pageSize": page_size,
        # "withSalaryMatch": "true",
        "salaryCurrency": "PLN",
        "salaryPeriod": "month",
        # "region": "pl",
        # "language": "pl-PL"
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        "Accept-Language": "pl-PL,pl;q=0.9"
    }
    response = httpx.get(url, params=params, headers=headers)
    return response.json()

offers = fetch_nofluffjobs_offers()
# print(f"Fetched {len(offers)} offers from NoFluffJobs")
print(offers['totalCount'])
print(len(offers['postings']))

21420
21420


### bulldogjob.pl

In [27]:
import httpx

def fetch_bulldogjob_offers():
    """Fetch all job offers from bulldogjob.pl GraphQL API."""
    url = "https://bulldogjob.pl/graphql"
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "*/*",
    }
    payload = {
        "operationName": "jobOffers",
        "variables": {
            "amount": 100,
            "language": "pl",
            "country": "PL"
        },
        "query": "query jobOffers($amount: Int, $language: LocaleEnum!, $country: String) {\n  jobOffers(amount: $amount, language: $language, country: $country) {\n    id\n    title\n    company { name }\n    city\n    salaryFrom\n    salaryTo\n    experienceLevel\n    remote\n    publishedAt\n    technologies { name }\n    employmentType\n    skills { name }\n    description\n    applyUrl\n  }\n}\n"
    }
    response = httpx.post(url, headers=headers, json=payload)
    return response.json()

offers = fetch_bulldogjob_offers()
print(offers)

{'errors': [{'message': "Field 'jobOffers' doesn't exist on type 'Query'", 'locations': [{'line': 2, 'column': 3}], 'path': ['query jobOffers', 'jobOffers'], 'extensions': {'code': 'undefinedField', 'typeName': 'Query', 'fieldName': 'jobOffers'}}, {'message': 'Variable $amount is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'amount'}}, {'message': 'Variable $language is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'language'}}, {'message': 'Variable $country is declared by jobOffers but not used', 'locations': [{'line': 1, 'column': 1}], 'path': ['query jobOffers'], 'extensions': {'code': 'variableNotUsed', 'variableName': 'country'}}]}


PRACUJ.pl

zapytanie grouped?languageCode **** zawiera odpowiedź

Zapytanie grouped prowadzi do:\
/jobOffers/listing/count/grouped?\
po zamianie na:\
/jobOffers/listing/grouped?\
poprawnie zwraca wyniki\
https://massachusetts.pracuj.pl/jobOffers/listing/count/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false&groupBy=ws&groupBy=et&groupBy=tc&groupBy=p&groupBy=ao&groupBy=wm&groupBy=ua&groupBy=wpl&groupBy=its&groupBy=itth&groupBy=ap&groupBy=hs\
\
https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false&groupBy=ws&groupBy=et&groupBy=tc&groupBy=p&groupBy=ao&groupBy=wm&groupBy=ua&groupBy=wpl&groupBy=its&groupBy=itth&groupBy=ap&groupBy=hs

In [ ]:
import httpx

def fetch_pracujpl_offers(url):
    """Fetch job offers from Pracuj.pl grouped endpoint."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Accept": "application/json, text/plain, */*",
        # "x-xsrf-token": xsrf_token
    }
    response = httpx.get(url, headers=headers)
    return response.json()["groupedOffers"]

source_url = "https://it.pracuj.pl/praca/praca%20zdalna;wm,home-office?et=17%2C4&ap=true&its=ai-ml%2Cbig-data-science&itth=37"

# Przykład użycia:
url = "https://massachusetts.pracuj.pl/JobOffers/listing/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false&groupBy=ws&groupBy=et&groupBy=tc&groupBy=p&groupBy=ao&groupBy=wm&groupBy=ua&groupBy=wpl&groupBy=its&groupBy=itth&groupBy=ap&groupBy=hs"
# xsrf_token = "TU_WSTAW_TOKEN"
offers = fetch_pracujpl_offers(url)
print(len(offers))

46


### cała procedura:
1: otwórz https://it.pracuj.pl/praca/praca%20zdalna;wm,home-office?et=17%2C4&ap=true&its=ai-ml%2Cbig-data-science&itth=37 \
2: z pierwszego requestu: https://massachusetts.pracuj.pl/jobOffers/listing/count?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false podmień /count na /grouped:
https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false

In [10]:
# Test długości - poprawny i wygenerowany
source_1 = "https://it.pracuj.pl/praca?et=17%2C18&ap=true&its=ai-ml%2Cbig-data-science%2Cfullstack%2Cbusiness-analytics&itth=37%2C77"
expected_1 = "https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=18&ap=true&its=ai-ml&its=big-data-science&its=fullstack&its=business-analytics&itth=37&itth=77&pn=1&rop=50&iwhpl=false"
print("test 1")
print(len(build_pracujpl_api_url(source_1)) == len(expected_1))
print(build_pracujpl_api_url(source_1))
print(expected_1)
source_2 = "https://it.pracuj.pl/praca/praca%20zdalna;wm,home-office?et=17%2C4&ap=true&its=ai-ml%2Cbig-data-science&itth=37"
expected_2 = "https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false"
print("test 2")
print(len(build_pracujpl_api_url(source_2)) == len(expected_2))
print(build_pracujpl_api_url(source_2))
print(expected_2)
source_3 = "https://it.pracuj.pl/praca/praca%20hybrydowa;wm,hybrid?et=4&sal=1&its=big-data-science%2Csystem-analytics&itth=33"
expected_3 = "https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=4&its=big-data-science&its=system-analytics&itth=33&wm=hybrid&salMin=1&salRate=0&pn=1&rop=50&iwhpl=false"
print("test 3")
print(len(build_pracujpl_api_url(source_3)) == len(expected_3))
print(build_pracujpl_api_url(source_3))
print(expected_3)

test 1
True
https://massachusetts.pracuj.pl/JobOffers/listing/grouped?et=17&et=18&ap=true&its=ai-ml&its=big-data-science&its=fullstack&its=business-analytics&itth=37&itth=77&subservice=1&pn=1&rop=50&iwhpl=false
https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=18&ap=true&its=ai-ml&its=big-data-science&its=fullstack&its=business-analytics&itth=37&itth=77&pn=1&rop=50&iwhpl=false
test 2
True
https://massachusetts.pracuj.pl/JobOffers/listing/grouped?wm=home-office&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&subservice=1&pn=1&rop=50&iwhpl=false
https://massachusetts.pracuj.pl/jobOffers/listing/grouped?subservice=1&et=17&et=4&ap=true&its=ai-ml&its=big-data-science&itth=37&wm=home-office&pn=1&rop=50&iwhpl=false
test 3
True
https://massachusetts.pracuj.pl/JobOffers/listing/grouped?wm=hybrid&et=4&salMin=1&salRate=0&its=big-data-science&its=system-analytics&itth=33&subservice=1&pn=1&rop=50&iwhpl=false
https://massachusetts.pracuj.pl/jobOffers/listing/g

# Wydobycie JSON z linku

In [9]:
# Tworzenie linku do JSON API z linku
import re
import urllib.parse
import httpx
import json

def parse_sal_param(key: str, value: str) -> list:
    """Parse 'sal' parameter to API format."""
    if key == "sal":
        return [("salMin", value), ("salRate", "0")]
    return []

def parse_wm_param(path: str) -> list:
    """Extracts 'wm' parameter from path."""
    parts = path.split('?')
    for part in parts:
        work_type_pattern = rf"^(.+?);wm,(.+)$"
        match = re.match(work_type_pattern, part)
        if match:
            wm_value = match.group(2)
            return [("wm", wm_value)]
    return []

def parse_et_param(key: str, value: str) -> list:
    """Parse 'et' parameter to API format."""
    if key == "et":
        return [("et", v) for v in value.split(',')]
    return []

def parse_its_param(key: str, value: str) -> list:
    """Parse 'its' parameter to API format."""
    if key == "its":
        return [("its", v) for v in value.split(',')]
    return []

def parse_itth_param(key: str, value: str) -> list:
    """Parse 'itth' parameter to API format."""
    if key == "itth":
        return [("itth", v) for v in value.split(',')]
    return []

def parse_other_param(key: str, value: str) -> list:
    """Parse other parameters to API format."""
    if key not in {"sal", "wm", "et", "its", "itth"}:
        return [(key, value)]
    return []

def parse_params_from_path(path: str) -> tuple[list, set]:
    """Parse parameters from URL path."""
    params = []
    keys_present = set()
    for part in path.split(';')[1:]:
        if '=' in part:
            key, value = part.split('=', 1)
            params += parse_sal_param(key, value)
            params += parse_et_param(key, value)
            params += parse_its_param(key, value)
            params += parse_itth_param(key, value)
            params += parse_other_param(key, value)
            keys_present.add(key)
    wm_params = parse_wm_param(path)
    if wm_params:
        params += wm_params
        keys_present.add("wm")
    return params, keys_present

def parse_params_from_query(query: dict) -> tuple[list, set]:
    """Parse parameters from URL query."""
    params = []
    keys_present = set()
    for key, value_list in query.items():
        value = value_list[0]
        params += parse_sal_param(key, value)
        params += parse_et_param(key, value)
        params += parse_its_param(key, value)
        params += parse_itth_param(key, value)
        params += parse_other_param(key, value)
        keys_present.add(key)
    return params, keys_present

def add_default_params(params: list, keys_present: set) -> None:
    """Add default parameters if missing."""
    defaults = [
        ("subservice", "1"),
        ("pn", "1"),
        ("rop", "50"),
        ("iwhpl", "false"),
    ]
    for key, value in defaults:
        if key not in keys_present:
            params.append((key, value))

def build_pracujpl_api_url(source_url: str) -> str:
    """Build API URL from source_url."""
    parsed_url = urllib.parse.urlparse(source_url)
    base_api = "https://massachusetts.pracuj.pl/JobOffers/listing/grouped"
    query = urllib.parse.parse_qs(parsed_url.query)
    path_only = source_url.split('/', 3)[-1].split('?', 1)[0]

    path_params, path_keys = parse_params_from_path(path_only)
    query_params, query_keys = parse_params_from_query(query)
    params = path_params + query_params
    keys_present = path_keys | query_keys

    add_default_params(params, keys_present)
    api_query = urllib.parse.urlencode(params, doseq=True)
    return f"{base_api}?{api_query}"

response = httpx.get(build_pracujpl_api_url("https://it.pracuj.pl/praca/praca%20zdalna;wm,home-office?et=17%2C4&ap=true&its=big-data-science%2Cbackend&itth=37"))
print(json.dumps(response.json(), indent=4, ensure_ascii=False))

{
    "groupedOffers": [
        {
            "technologies": [
                "Python",
                "LLM",
                "Microsoft Azure"
            ],
            "aboutProjectShortDescription": "We are seeking an experienced Python Developer to join a dynamic project focused on leveraging Azure and LLM technologies., , You will be working in a highly collaborative environment to develop and maintain Python-based solutions, integrating...",
            "groupId": "dc120000-56be-0050-8bbe-08ddeedd872b",
            "jobTitle": "Python Developer",
            "companyName": "SQUARE ONE RESOURCES sp. z o.o.",
            "companyProfileAbsoluteUri": "https://pracodawcy.pracuj.pl/company/20409928",
            "companyId": 20409928,
            "companyLogoUri": "https://logos.gpcdn.pl/loga-firm/20409928/00230000-56be-0050-c213-08dd5a4ccc03_280x280.png",
            "lastPublicated": "2025-09-08T14:22:14.1Z",
            "expirationDate": "2025-10-08T21:59:59Z",
            "sa